In [ ]:
from langchain_community.document_loaders import TextLoader , PyPDFLoader , AmazonTextractPDFLoader , UnstructuredPDFLoader , DirectoryLoader , WebBaseLoader
from langchain_huggingface import ChatHuggingFace , HuggingFaceEndpoint
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import PromptTemplate
import os

USER_AGENT environment variable not set, consider setting it to identify your requests.


# Text document loader

In [12]:
loader = TextLoader( file_path=r"D:\machine learning\lang chain\document loader\document.txt" )

doc = loader.load()

print((doc[0]))

page_content='Homosexuality refers to romantic or sexual attraction between people of the same sex and is a natural variation of human sexuality that has existed across cultures and throughout history. Same-sex marriage is the legal recognition of a committed partnership between two people of the same sex, granting them the same rights, responsibilities, and social recognition as opposite-sex couples. Supporters of same-sex marriage emphasize principles of equality, love, and personal freedom, arguing that consenting adults should have the right to marry regardless of gender. Over time, increasing social acceptance and legal reforms in many countries have reflected a broader understanding of human rights and the importance of respecting diverse identities and families.
' metadata={'source': 'D:\\machine learning\\lang chain\\document loader\\document.txt'}


In [18]:
prompt = PromptTemplate(
    template='Write a 5 points on the following text - {text}',
    input_variables=['text']
)

parser = StrOutputParser()

In [ ]:
llm = HuggingFaceEndpoint(
    repo_id='deepseek-ai/DeepSeek-V3.2',
    temperature=0.5,
    # max_new_tokens=128,
    huggingfacehub_api_token=os.environ["HF_TOKEN"]
)

model = ChatHuggingFace(llm = llm)

In [20]:
chain = prompt | model | parser

result = chain.invoke({'text' : doc[0].page_content})

In [21]:
result

'Here are 5 key points based on the provided text:\n\n1.  **Definition of Homosexuality:** It is defined as romantic or sexual attraction between people of the same sex, described as a natural and historically consistent variation of human sexuality.\n2.  **Definition and Purpose of Same-Sex Marriage:** It is the legal recognition of a same-sex partnership, specifically intended to grant equal rights, responsibilities, and social status as those afforded to opposite-sex marriages.\n3.  **Core Arguments for Support:** Advocacy for same-sex marriage is rooted in the principles of **equality, love, and personal freedom**, with the central claim that consent'

# PyPDF document loader

In [26]:
pdf_loader = PyPDFLoader(file_path=r"D:\OneDrive - Indian Institute of Technology Indian School of Mines Dhanbad\Documents\5th sem\ugc\Explosives_and_Blasting_Notes .pdf")

pdf_doc = pdf_loader.load()

In [34]:
len(pdf_doc)

4

# Directory loader

In [78]:
loader = DirectoryLoader(
    path=r"D:\OneDrive - Indian Institute of Technology Indian School of Mines Dhanbad\Downloads\New folder",
    glob='*.pdf',
    loader_cls=PyPDFLoader
)

docs_generator = loader.lazy_load()
docs = []

for document in docs_generator : 
    docs.append(document)

In [81]:
docs[0].metadata

{'producer': 'iLovePDF',
 'creator': 'PyPDF',
 'creationdate': '2025-07-31T19:03:31+05:30',
 'author': 'Swapnil Mishra',
 'title': 'Microsoft PowerPoint - Unit 1_30.07.25',
 'moddate': '2025-07-31T13:36:53+00:00',
 'source': 'D:\\OneDrive - Indian Institute of Technology Indian School of Mines Dhanbad\\Downloads\\New folder\\Unit_1_17539690771511218071688b71b5060c7.pdf',
 'total_pages': 56,
 'page': 0,
 'page_label': '1'}

# Document Loader

In [90]:
url1 = 'https://www.amazon.com/Designing-Data-Intensive-Applications-Reliable-Maintainable/dp/1449373321/?_encoding=UTF8&ref_=pd_hp_d_btf_ci_mcx_mr_ca_id_hp_d'
url2 = 'https://www.amazon.com/System-Design-Interview-insiders-Second/dp/B08CMF2CQF/ref=bmx_dp_d_sccl_1_2/135-9646303-1580503?psc=1'
url3 = 'https://www.amazon.com/dp/1835460038/ref=sspa_dk_detail_3?psc=1&sp_csd=d2lkZ2V0TmFtZT1zcF9kZXRhaWxfdGhlbWF0aWM'
web_loader = WebBaseLoader([url1 , url2 , url3])

docs = web_loader.load()

In [94]:
docs[0].page_content.replace('\n' , '')

'Designing Data-Intensive Applications: The Big Ideas Behind Reliable, Scalable, and Maintainable Systems: Kleppmann, Martin: 9781449373320: Amazon.com: BooksSkip to        Main content              About this item              About this item              About this item              Buying options              Compare with similar items              Videos              Reviews            Keyboard shortcuts  Searchalt+/Cartshift+alt+CHomeshift+alt+HOrdersshift+alt+OAdd to cartshift+alt+KShow/Hide shortcutsshift+alt+ZTo move between items, use your keyboard\'s up or down arrows..us                   Deliver to                                   India                BooksSelect the department you want to search inAll DepartmentsArts & CraftsAutomotiveBabyBeauty & Personal CareBooksBoys\' FashionComputersDealsDigital MusicElectronicsGirls\' FashionHealth & HouseholdHome & KitchenIndustrial & ScientificKindle StoreLuggageMen\'s FashionMovies & TVMusic, CDs & VinylPet SuppliesPrime VideoSof

In [99]:
import re

prompt = PromptTemplate(
    template='Answer the following questions based on the given text \n questions - {questions} \n\n text - {text}',
    input_variables=['questions' , 'text']
)

chain = prompt | model | parser

result = chain.invoke({
    'text' : re.sub(r'\s+', ' ', docs[0].page_content.replace('\n' , '')).strip(),
    'questions' :  'Which production is discussed in the text ? \n what is the price of the product? \n is it benificial to buy this product ?'
})

In [100]:
print(result)

Based on the text, here are the answers to your questions:

**1. Which production is discussed in the text?**  
The text is about the book **"Designing Data-Intensive Applications: The Big Ideas Behind Reliable, Scalable, and Maintainable Systems"** by Martin Kleppmann.

**2. What is the price of the product?**  
The listed price for the paperback edition is **INR 3,349.87** (with 38% savings from the list price of INR 5,431.31). Other formats like Kindle and Audio CD have different prices.

**3. Is it beneficial to buy this product?**  
Yes, the text strongly suggests it is beneficial. It is described as a **#1 Best Seller** in Data Modeling & Design, with a rating of 4.8 out of 5 stars from over 5,400 reviews. The content is praised as a comprehensive, practical, and insightful guide for software engineers and architects dealing with data systems, making it valuable for learning and professional growth.
